# Notebook 04 — CVaR Optimization

**Purpose:** Run the efficient frontier using CVaR as the optimization objective. Output a set of portfolios ranging from minimum risk to maximum yield.

**Inputs:** 
- `data/processed/risk_adjusted_returns.csv`
- `data/processed/cov_normal.csv`
- `data/processed/cov_stress.csv`
- `data/processed/cov_blended.csv`
- `data/processed/expected_returns.csv` (for scenarios)

**Outputs:** 
- `data/processed/frontier_portfolios.csv`
- `data/processed/named_portfolios.json`

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import cvxpy as cp
import json

# Add src to path
sys.path.append(os.path.abspath('../src'))

import optimizer as opt
import utils

np.random.seed(42)

## 1. Load Data

We need the risk-adjusted expected returns and the blended covariance matrix.

In [ ]:
risk_adj_df = pd.read_csv('../data/processed/risk_adjusted_returns.csv')
cov_blended = pd.read_csv('../data/processed/cov_blended.csv', index_index='instrument' if 'instrument' in pd.read_csv('../data/processed/cov_blended.csv').columns else 0)
if 'instrument' in cov_blended.columns:
    cov_blended = cov_blended.set_index('instrument')
else:
    cov_blended.index = cov_blended.columns

# Ensure order matches risk_adj_df
instruments = risk_adj_df['instrument'].tolist()
mu = risk_adj_df.set_index('instrument').loc[instruments, 'risk_adjusted_yield_idr'].values
S = cov_blended.loc[instruments, instruments].values

# Scenarios from historical returns
hist_returns = pd.read_csv('../data/processed/expected_returns.csv')
pivot_hist = hist_returns.pivot(index='date', columns='instrument', values='yield_idr').reindex(columns=instruments).ffill().dropna()
returns_scenarios = pivot_hist.values

## 2. Define Constraints

- Sum of weights = 1
- No shorting (weights >= 0)
- Max 50% in any single instrument
- Pendle YT max 15%

In [ ]:
n = len(instruments)
w = cp.Variable(n)

constraints = [
    cp.sum(w) == 1,
    w >= 0,
    w <= 0.50
]

# Find Pendle YT index
if 'pendle_yt' in instruments:
    yt_idx = instruments.index('pendle_yt')
    constraints.append(w[yt_idx] <= 0.15)

print("Constraints defined.")

## 3. Generate Frontier

Minimize CVaR for 100 points.

In [ ]:
frontier = opt.run_frontier(mu, S, constraints, returns_scenarios, n_points=100)

frontier_df = pd.DataFrame(frontier)
for i, inst in enumerate(instruments):
    frontier_df[f'w_{inst}'] = frontier_df['weights'].apply(lambda x: x[i])

frontier_df.to_csv('../data/processed/frontier_portfolios.csv', index=False)
print(f"Frontier generated with {len(frontier_df)} portfolios.")

## 4. Named Portfolios

In [ ]:
min_risk = frontier_df.iloc[0].to_dict()
max_yield = frontier_df.iloc[-1].to_dict()
balanced = frontier_df.iloc[len(frontier_df)//2].to_dict()

named = {
    "min_risk": min_risk,
    "balanced": balanced,
    "max_yield": max_yield
}

with open('../data/processed/named_portfolios.json', 'w') as f:
    json.dump(named, f, indent=2)

print("Named portfolios saved.")